In [1]:
import numpy as np
from itertools import combinations
from validation import _validate_operator, _validate_basis, _validate_stabiliser_set
from dictionaries import pauli_prod_dict, sdict, hdict, cnotdict


### Introduction

The following codebook is designed to guide one through the building of an efficient classical simulator for n-qubit systems using the stabiliser formalism and Gottesman-Knill theorem. 

### Reprentation of a Quantum State

The following codebook has implemented the following conventions and definitions for various objects. The examples given will correspond to what that object may look like when simulating a 2, 3, 4 qubit system. (except stabiliser set this only has a 3)

`pauli`: A singular character, stored as a string. It must be one of `"I"`, `"X"`, `"Y"`, or `"Z"`.
> e.g. `"X"`, `"Z"`, `"Y"`

`paulistring`: A string of many individual paulis, whose length matches the number of qubits we are simulating. Each pauli represents an operation on the qubit corresponding to its position in the string. 
> e.g. `"XX"`, `"YIY"`, `"XIZY"`

`phase`: A complex integer to represent the phase of a stabiliser or operator. It must be one of `1`, `-1`, `1j`, or `-1j`
> e.g. `1j`, `-1`, `1`

`stabiliser`: A list containing one phase and one pauli string. Stabilisers cannot have complex phases. 
> e.g. `[1, "XZ"]`, `[-1, "IXX"]`, `[1, "ZYXI"]`

`stabiliser_set`: A list whose elements are all stabiliser sets. A stabiliser set on n qubits must contain n stabilisers to uniquely identify a set. Further all stabilisers in a set must mutually commute. 
> e.g. `[[1, "XZ"],[-1, "YX"]]`

> e.g. `[[-1, "IIX"],[1, "IXI"],[1, "XII"]]`

> e.g. `[[1, "IZZI"],[-1, "XIIX"],[-1, "IYYI"],[1, "YIIY"]]`

These make up the basic building blocks necessary to represent a quantum state. 




Recall that mathematically, a stabiliser set, S, must satisfy three hard conditions:

- All elements must must be Hermitian Pauli observalbles.

- All elements of the set must mutually commute.

- The negative identity may not appear as an element of the stabiliser group. 

We will design the second of these only, as the other conditions require a more advanced representation of the stabiliser formalism. The remaining checks have already been built, and have been imported from `validation.py`.

Below, `_commutation_1qubit` will accept two Paulis and return a parity, corresponding to whether the Paulis commute or anticommute. `_commutation` should then use this, to determine whether two stabilisers commute. 

In [2]:
def _commutation_1qubit(pauli1, pauli2):
    comm_string = pauli1+pauli2
    if "I" in comm_string:
        return 1
    elif pauli1 == pauli2:
        return 1
    else:
        return -1

def _commutation(stab1, stab2):
    parity = 1
    for i in range(len(stab1[1])):
        parity = parity * _commutation_1qubit(stab1[1][i], stab2[1][i])
    return parity

### Operations on Quantum States

`operator`: There are two types of operators, which can either be the tensor product of 1-qubit operations, or a single 2-qubit entangling operation. These are formatted in lists, with the first element being the phase of the operation and the second being a string to represent the operation itself.

>e.g. `[-1j, "XS"]`, `[1, "YHZ"]`, `[-1, "CNOT_IIZX"]`

The string element can take one of two formats:

**1 qubit operators**: The string element must have length equal to the number of qubits, and whose elements are either a pauli, or either of `"H"` and `"S"`.
>e.g. `"HS"`, `"IXY"`, `"ISZH"`

**Entangling operator**: The string element must start with `"CNOT_"`, to signify an entangling gate, and be followed by a string of length corresponding to the number of qubits. This latter string may contain exactly one `"C"` and one `"T"`. These represent the CNOT operation utilising a control and target qubit, corresponding to the positions of `"C"` and one `"T"` respectively. All other elements of this string must be identities, `"I"`. 
>e.g. `"CNOT_CT"`, `"CNOT_CIT"`, `"CNOT_ITIC"`

A crucial operation we need to be able to perform is to determine the product of two stabilisers. A dictionary mapping all products as strings has been provided for you in `dictionaries.py`, and imported as `pauli_prod_dict`. The dictionary is indexed by two paulis concatenated into a single string (e.g. X*Y is indexed as `"XY"`) and will return the phase and resulting pauli as a tuple. 

In [3]:
def _stabiliser_product(stabiliser1, stabiliser2):
    """Calculates the direct matrix product between two stabiliser-type objects."""
    product = ""
    phase = stabiliser1[0]*stabiliser2[0]
    for i in range(len(stabiliser1[1])):
        i_phase, i_prod = pauli_prod_dict[stabiliser1[1][i]+stabiliser2[1][i]]
        product += i_prod
        phase *= i_phase
    return phase, product

As we know, another key operation we need to perform is conjugation of our stabilisers by arbitrary cliffords. This allows us to evolve the state by the operator in the stabiliser formalism. 

The helper, `_conj_pauli`, should compute the conjugation of a single Pauli by another Pauli. `_conj_clifford` should then iterate over a the Pauli string component of a stabiliser to find the conjugation by an arbitrary 1-qubit tensored clifford unitary. The dictionaries `hdict` and `sdict` are provided to map a between Paulis under conjugation by respective cliffords. Once again, these will return both phase and a Pauli. 

In [4]:
def _conj_pauli(pauli, op):
    """Takes a pauli gate as a string and conjugates it with another pauli gate. """
    if op == pauli:
        return 1, op
    else: 
        return -1, op

def _conj_clifford(stabiliser, operator):
    pauli_string = stabiliser[1]
    new_phase = stabiliser[0]
    new_string = ""
    for i in range(len(pauli_string)):

        if operator[1][i] in ("X","Y","Z"):
            add_phase, add_string = _conj_pauli(pauli_string[i], operator[1][i])
        elif operator[1][i]=="H":
            add_phase, add_string = hdict[pauli_string[i]]
        elif operator[1][i]=="S":
            add_phase, add_string = sdict[pauli_string[i]]

        new_phase *= add_phase
        new_string += add_string

    return new_phase, new_string

Finally, we need to be be able to conjugate a stabiliser by a 2-qubit entangling gate. Once again, `cnotdict` is provided for the mappings between 2-qubit Pauli strings under CNOT conjugation. 

hint 1: entangling operators have a leading string `"CNOT_"` to sign themselves. Remember that only the end of the operator's string actually contains the operator information. 

In [5]:
def _conj_cnot(stabiliser, cnot_op):
    pauli_phase = stabiliser[0]
    pauli_string = stabiliser[1]
    
    # remove leading CNOT_, find positions of control and target, and identify string "CT"
    cnot_op =  (cnot_op[0], cnot_op[1].removeprefix("CNOT_"))
    control = cnot_op[1].find("C")
    target = cnot_op[1].find("T")
    ct_qubits = pauli_string[control]+pauli_string[target]

    # if there is only one I, take conjugate from dict. 
    phase, ct_qubits = cnotdict[ct_qubits]
    pauli_phase *= phase

    # insert qubits back into string
    pauli_string = pauli_string[:control] + ct_qubits[0] + pauli_string[control+1:]
    pauli_string = pauli_string[:target] + ct_qubits[1] + pauli_string[target+1:]
    return phase, pauli_string


With these operations, we can now define arbitrary clifford operations on stabiliser sets. We have `transform_stabiliser` which will operate on a stabiliser and evolve it by conjugation .

hint: be careful to track the phase of each stabiliser correctly.


In [6]:
def transform_stabiliser(stabiliser, operator):
    if operator[1].startswith("CNOT_"):
        new_phase, transformed_string = _conj_cnot(stabiliser, operator)
    else: 
        new_phase, transformed_string = _conj_clifford(stabiliser, operator)
    return new_phase, transformed_string

def transform_state(n_qubits, stabiliser_set, operator):
    # validate the operator
    _validate_operator(n_qubits, operator)

    # perform the transformation
    for i,stabiliser in enumerate(stabiliser_set):
        stabiliser_set[i] = transform_stabiliser(stabiliser,operator)
    return stabiliser_set

### Measurement of Quantum States

`basis`:

In [7]:
def _comm_measurement():
    pass

def _1_anti_comm_measurement():
    pass

def _many_anti_comm_measurement():
    pass

def measure_state(stabiliser_set, basis):
    _validate_basis(basis)


### Testing our Simulator

Now, we have defined how we represent quantum states, how we can perform operations on these states and finally how to read out observables and update the resulting state. This is everything we need for a complete simulator for all pure states that can be described by the stabiliser formalism. 

We will now bundle our functions together into a class and demonstrate that these functions indeed scale as O[poly(n)] as desired.

In [8]:
class pure_state:
    def __init__(self):
        self.valid_state = None
        self.n_qubits = None
        self.stab_group = None

    def prepare_state_direct(self, stabiliser_set):
        self.n_qubits = len(stabiliser_set)
        self.valid_state = _validate_stabiliser_set(stabiliser_set)   
        self.stabiliser_set = stabiliser_set
        
    def _stabiliser_product(stabiliser1, stabiliser2):
        return _stabiliser_product(stabiliser1, stabiliser2)

    def prepare_state_transform(self, stabiliser_set, operator):
        pass

    def transform_state(self, operator):
        self.stabiliser_set = transform_state(self.n_qubits, self.stabiliser_set, operator)
        try: 
            self.valid_state = self._validate_stabiliser_set(self.stabiliser_set)
        except ValueError:
            raise ValueError("Transformation produced unexpected state. Could not validate output state.")

    def measure_state(self):
        pass



### Mixed States

Mixed states can likewise be efficiently represented using the stabiliser formalism. Ad hoc, we would simply store many stabiliser sets, alongisde classical probabilities, corresponding to preparation uncertainty. Operations can be performed on all stabiliser sets to update the mixture. Any measurement need only take the outcomes of the individual sets and weigh these by the mixture to return a probabilistic outcome. 


In [9]:
class mixed_state:
    def __init__(self):
        pass

### Additional Code

In [10]:
string_to_bin_dict = {"I":(0,0),
                      "Z":(0,1),
                      "X":(1,0),
                      "Y":(1,1)
                      }

def stab_to_bin(stab):
    if stab[0]==1:
        phase = 0
    else: 
        phase = 1
    vec_x = []
    vec_z = []
    for pauli in stab[1]:
        x_i = string_to_bin_dict[pauli]
        vec_x.append(x_i[0])
        vec_z.append(x_i[1]) 
    return phase,vec_x+vec_z


In [11]:
# math for testing
def dagger(matrix):
    return matrix.conj().T

i = np.identity(2)
x = np.array([[0,1],[1,0]])
y = np.array([[0,-1j],[1j,0]])
z = np.array([[1,0],[0,-1]])
cnot = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]])
h = 1/np.sqrt(2)*np.array([[1,1],[1,-1]])
s = np.array([[1,0],[0,-1j]])